In [4]:
import pandas as pd
from whatsapp import WhatsappFilrt

# Instantiate & preprocess
wf = WhatsappFilrt()
df = wf.PreProcessing("Chat.txt")

# Ensure text column is formatted cleanly
df["Text"] = df["Text"].fillna("").astype(str)

print("=" * 40)
print("1 & 2. MOST ACTIVE DAY & TIME")
print("=" * 40)
most_active_date = df["Date"].value_counts().idxmax()
most_active_time = df["Time"].value_counts().idxmax()

print(f"• Most Active Date: {most_active_date}")
print(f"• Most Active Time: {most_active_time}\n")


print("=" * 40)
print("3. MEDIA COUNT SENT BY EACH PERSON")
print("=" * 40)
# Identify media messages (<media omitted>)
media_df = df[df["Text"].str.contains("<media omitted>", case=False, na=False)]

if not media_df.empty:
    media_by_person = media_df["Name"].value_counts().reset_index()
    media_by_person.columns = ["Person", "Media Sent Count"]
    print(media_by_person.to_string(index=False))
else:
    print("No media found in this chat log.")
print("\n")


print("=" * 40)
print("4. MISSED CALLS (VOICE & VIDEO)")
print("=" * 40)
# Filter for missed voice or video calls
missed_calls = df[df["Text"].str.contains("missed voice call|missed video call", case=False, na=False)]

if not missed_calls.empty:
    print(f"Total Missed Calls: {len(missed_calls)}\n")
    print(missed_calls[["Date", "Time", "Name", "Text"]].to_string(index=False))
else:
    print("No missed calls detected.")

Execution Time: 0.00013136863708496094
1 & 2. MOST ACTIVE DAY & TIME
• Most Active Date: 16/12/2022
• Most Active Time: 5:44 pm

3. MEDIA COUNT SENT BY EACH PERSON
                Person  Media Sent Count
             msranjith               127
      +971 50 152 6671               116
       +91 96551 11513                39
       +91 86080 66773                34
              ganeshan                32
               Raguram                30
        Sathish Police                30
                Urunda                17
          Gopiguna Rmm                17
Venkat Raghavan School                14
       +91 86820 64983                10
       Manikandaninsur                 7
                    Mm                 4
         Sanmuga kumar                 3
                Kannan                 3
       +91 78718 33139                 2
    Rajkumar Ranjitham                 2
       +91 94981 05309                 2
                  Kosu                 1
       +91 70101

In [5]:
import pandas as pd

# 1. Instantiate class and load chat data
wf = WhatsappFilrt()
df = wf.PreProcessing("Chat.txt")

# Ensure text and name columns are string format
df["Text"] = df["Text"].fillna("").astype(str)
df["Name"] = df["Name"].fillna("Unknown").astype(str).str.lower().str.strip()

# =========================================================
# 1. TALKATIVE & LESS TALKATIVE PARTICIPANTS
# =========================================================
message_counts = df["Name"].value_counts()

most_talkative = message_counts.idxmax().upper()
least_talkative = message_counts.idxmin().upper()

print("=" * 50)
print("TALKATIVE METRICS")
print("=" * 50)
print(f"• Most Talkative Person : {most_talkative} ({message_counts.max()} messages)")
print(f"• Less Talkative Person : {least_talkative} ({message_counts.min()} messages)\n")

print("Message breakdown per person:")
print(message_counts.to_string())
print("\n")


# =========================================================
# 2. ROW-BY-ROW FLIRT MODEL CHECKING
# =========================================================
# Defined flirt keywords from your WhatsappFilrt class
flirt_words = [
    'kiss', 'hug', 'date', 'cute', 'beautiful', 'sexy', 'hot', 
    'adorable', 'uma', 'darling', 'fuck', 'porn', 'x', 'sex', 
    'matter', 'nipple', 'virgin', 'sperm', 'seduce', 'condom', 'kk'
]

# Row-by-row text clean up & detection function
def analyze_row_flirt(text):
    # Remove emojis using your class method
    clean_text = wf.give_emoji_free_text(str(text)).lower()
    words = clean_text.split()
    
    # Check intersection with flirt_words
    matches = [word for word in words if word in flirt_words]
    
    is_flirt = len(matches) > 0
    return is_flirt, len(matches), ", ".join(matches)

# Apply row-by-row evaluation
flirt_results = df["Text"].apply(analyze_row_flirt)

# Assign results to DataFrame columns
df["Is_Flirt"] = [res[0] for res in flirt_results]
df["Flirt_Word_Count"] = [res[1] for res in flirt_results]
df["Detected_Flirt_Words"] = [res[2] for res in flirt_results]


# =========================================================
# 3. DISPLAY FLIRT RESULTS
# =========================================================
print("=" * 50)
print("FLIRT DETECTION SUMMARY")
print("=" * 50)

# Summary by Person
flirt_summary = df.groupby("Name").agg(
    Total_Messages=("Text", "count"),
    Flirty_Messages=("Is_Flirt", "sum"),
    Total_Flirt_Words=("Flirt_Word_Count", "sum")
).reset_index()

flirt_summary["Flirt_Message_%"] = round(
    (flirt_summary["Flirty_Messages"] / flirt_summary["Total_Messages"]) * 100, 2
)

print(flirt_summary.to_string(index=False))
print("\n")

print("=" * 50)
print("ROW-BY-ROW FLIRTY MESSAGES DETECTED")
print("=" * 50)

flirty_rows = df[df["Is_Flirt"] == True][["Date", "Time", "Name", "Text", "Detected_Flirt_Words"]]

if not flirty_rows.empty:
    print(f"Total Flirty Rows Found: {len(flirty_rows)}\n")
    print(flirty_rows.to_string(index=False))
else:
    print("No flirty messages detected in any row.")

TALKATIVE METRICS
• Most Talkative Person : MSRANJITH (397 messages)
• Less Talkative Person : URUNDA CREATED GROUP "NAMMA ISCHOOOL" (1 messages)

Message breakdown per person:
Name
msranjith                                                                                                        397
+971 50 152 6671                                                                                                 357
sathish police                                                                                                   137
kannan                                                                                                           128
urunda                                                                                                           122
ganeshan                                                                                                         120
+91 96551 11513                                                                                                  117